<a href="https://colab.research.google.com/github/AshantiVilladiego/FlyRankAI-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshantiVilladiego/FlyRankAI-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule (in plain words):** A page requires a title and meta description review if its **measured** average position is on Page 1 (Position 1-10), it **observes** decent search volume (>50 impressions), but its click-through rate (CTR) is terrible (<2%).

**Reason Code Output:** `page1_high_vol_low_ctr`

In [4]:
import pandas as pd
import numpy as np
import os
from google.colab import userdata

# 1. Load the data
hf_token = userdata.get('hf_token')
df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03",
    storage_options={"token": hf_token}
)

# Filter for usable search data only
df = df[df['gsc_impressions'] > 0].copy()
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']

print("--- SIGNAL 1: CTR vs POSITION ---")
# Bucket position into Page 1 (Top 3), Page 1 (Bottom), Page 2, Page 3+
bins = [0, 3.5, 10.5, 20.5, 1000]
labels = ['1. Top 3', '2. Pos 4-10', '3. Page 2', '4. Deep']
df['pos_bucket'] = pd.cut(df['gsc_avg_position'], bins=bins, labels=labels)

sig1 = df.groupby('pos_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()
print(sig1)
print("\nVerdict: CONFIRMED. Position heavily dictates baseline expected CTR.")

print("\n--- SIGNAL 2: VOLUME vs CTR (Page 1 only) ---")
# Does high impression volume mean lower CTR on page 1?
df_page1 = df[df['gsc_avg_position'] <= 10].copy()
df_page1['vol_bucket'] = pd.qcut(df_page1['gsc_impressions'], q=3, labels=['Low', 'Med', 'High'])

sig2 = df_page1.groupby('vol_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()
print(sig2)
print("\nVerdict: MIXED. High volume does not guarantee a terrible CTR, making low-CTR high-volume pages a fixable anomaly.")

--- SIGNAL 1: CTR vs POSITION ---
    pos_bucket        n   avg_ctr
0     1. Top 3   684726  0.004783
1  2. Pos 4-10  1372745  0.003396
2    3. Page 2   498758  0.002740
3      4. Deep   891643  0.001276

Verdict: CONFIRMED. Position heavily dictates baseline expected CTR.

--- SIGNAL 2: VOLUME vs CTR (Page 1 only) ---
  vol_bucket       n   avg_ctr
0        Low  759783  0.004950
1        Med  701774  0.003208
2       High  721927  0.003467

Verdict: MIXED. High volume does not guarantee a terrible CTR, making low-CTR high-volume pages a fixable anomaly.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# 1. Define the Rule conditions
is_page_1 = (df['gsc_avg_position'] <= 10).astype(int)
has_volume = (df['gsc_impressions'] > 50).astype(int)
terrible_ctr = (df['ctr'] < 0.02).astype(int)

# 2. Encode Action Label and Reason Code
df['action_label'] = np.where(is_page_1 * has_volume * terrible_ctr, 'Rewrite_Title_Meta', 'None')
df['reason_code'] = np.where(df['action_label'] == 'Rewrite_Title_Meta', 'page1_high_vol_low_ctr', 'n/a')

# 3. Create the Score (Rank by potential lost clicks: Impressions * (Expected 5% CTR - Actual CTR))
# This acts as a directional, decision-support score to prioritize the worst offenders.
df['score'] = np.where(
    df['action_label'] == 'Rewrite_Title_Meta',
    df['gsc_impressions'] * (0.05 - df['ctr']),
    0
)

# 4. Sort and isolate the ranked queue
queue = df[df['action_label'] != 'None'].sort_values('score', ascending=False).copy()
queue_export = queue[['report_date', 'client_hash_id', 'content_hash_id', 'action_label', 'reason_code', 'score', 'gsc_avg_position', 'gsc_impressions', 'ctr']]

# 5. Write to CSV
os.makedirs('work/outputs', exist_ok=True)
queue_export.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue generated! {len(queue_export)} items flagged for CTR fixes.")
print("\nTop 20 items for manual review:")
print(queue_export.head(20)[['content_hash_id', 'score', 'gsc_avg_position', 'ctr', 'gsc_impressions']])

Queue generated! 679152 items flagged for CTR fixes.

Top 20 items for manual review:
                  content_hash_id    score  gsc_avg_position       ctr  \
9655081  content_44f34c0a90047651  2003.20          0.083350  0.000025   
103612   content_34a70fea29d15f24  1948.15          2.764916  0.000051   
103661   content_945d6ff91386c817  1868.40          8.613948  0.000000   
8054586  content_eadb33b5df496f4a  1713.25          2.197507  0.006411   
8882581  content_fec55986a1868d62  1669.15          0.181500  0.000000   
9179913  content_eadb33b5df496f4a  1650.80          2.195988  0.007051   
8550024  content_44f34c0a90047651  1647.90          0.132532  0.000000   
8865290  content_44f34c0a90047651  1635.80          0.142508  0.000061   
9451315  content_fec55986a1868d62  1573.60          0.083407  0.000000   
7822469  content_44f34c0a90047651  1547.20          0.117814  0.000032   
8972299  content_eadb33b5df496f4a  1545.20          2.188397  0.006355   
7767282  content_44f34c0a9

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

*   **Ranks 1-10 (The extreme outliers):**
    *   **Action:** Rewrite_Title_Meta
    *   **Reason Code:** `page1_high_vol_low_ctr`
    *   **Confidence Note:** High. These items have massive impressions and near-zero clicks despite ranking well.
    *   **What would make it wrong:** If these URLs are informational utility pages (like "What time is it in Tokyo?"), Google's snippet answers the user's question directly on the search page. Zero clicks are expected, and rewriting the title won't change user behavior.

*   **Ranks 11-20 (The borderline picks):**
    *   **Action:** Rewrite_Title_Meta
    *   **Reason Code:** `page1_high_vol_low_ctr`
    *   **Confidence Note:** Medium. The impression volume is slightly lower, meaning a few lucky clicks could drastically alter the CTR percentage.
    *   **What would make it wrong:** The search intent might be highly visual (e.g., "living room layout ideas"). If our page is text-heavy, users will click Google Images instead of our link. The title isn't the problem; the content format is.

In [8]:
# Displaying the top 20 items discussed in the review above
print("--- TOP 20 ITEMS FOR REVIEW ---")
print(queue_export.head(20)[['content_hash_id', 'score', 'gsc_avg_position', 'ctr', 'gsc_impressions']])

--- TOP 20 ITEMS FOR REVIEW ---
                  content_hash_id    score  gsc_avg_position       ctr  \
9655081  content_44f34c0a90047651  2003.20          0.083350  0.000025   
103612   content_34a70fea29d15f24  1948.15          2.764916  0.000051   
103661   content_945d6ff91386c817  1868.40          8.613948  0.000000   
8054586  content_eadb33b5df496f4a  1713.25          2.197507  0.006411   
8882581  content_fec55986a1868d62  1669.15          0.181500  0.000000   
9179913  content_eadb33b5df496f4a  1650.80          2.195988  0.007051   
8550024  content_44f34c0a90047651  1647.90          0.132532  0.000000   
8865290  content_44f34c0a90047651  1635.80          0.142508  0.000061   
9451315  content_fec55986a1868d62  1573.60          0.083407  0.000000   
7822469  content_44f34c0a90047651  1547.20          0.117814  0.000032   
8972299  content_eadb33b5df496f4a  1545.20          2.188397  0.006355   
7767282  content_44f34c0a90047651  1537.55          0.088955  0.000065   
942680

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Identified:**
A major weakness in this baseline is that it blindly penalizes low Google Search Console clicks without checking alternative traffic sources. If a page has low GSC clicks but **observes** high traffic in the `sessions_ai` (Gemini/ChatGPT) or `sessions_social` columns, the user *is* getting our content—just not via a traditional Google Search link. Flagging these successful pages as "broken" is a false positive.

**Leakage Check:**
Confirmed. No future data was used. The baseline scores are calculated using strictly **measured** daily metrics (`gsc_impressions`, `gsc_avg_position`, `gsc_clicks`) occurring on the exact same `report_date`. There are no trailing or leading window aggregates that could cause temporal leakage, and no proprietary product flags were reverse-engineered.

In [11]:

weak_picks = df[(df['action_label'] == 'Rewrite_Title_Meta') & (df['sessions_ai'] > 10)]

print(f"Found {len(weak_picks)} 'Weak Picks' in our queue.")
if len(weak_picks) > 0:
    print("Example of a false positive (High AI traffic, penalized for low GSC clicks):")
    print(weak_picks[['content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ctr', 'sessions_ai']].head(3))
else:
    print("No immediate AI traffic false positives found in this exact slice, but the risk remains.")

Found 0 'Weak Picks' in our queue.
No immediate AI traffic false positives found in this exact slice, but the risk remains.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.